In [0]:


%sql
SELECT 
    
    new_id,
    franchiseID,
    RANK() OVER (PARTITION BY new_id ORDER BY franchiseID DESC) as dept_rank,
    LAG(franchiseID, 1) OVER (ORDER BY review_date) as prev_franchiseID,
    LEAD(franchiseID, 1) OVER (ORDER BY review_date) as next_franchiseID
FROM samples.bakehouse.media_customer_reviews

In [0]:
%sql
use catalog `samples`; select * from `bakehouse`.`media_customer_reviews` limit 100;

In [0]:
from pyspark.sql.functions import lag, lead
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from datetime import date

# Create sample data with date and price columns
data = [
    ("Product A", date(2024, 1, 1), 100),
    ("Product A", date(2024, 1, 2), 105),
    ("Product A", date(2024, 1, 3), 103),
    ("Product A", date(2024, 1, 4), 110)
]
df_prices = spark.createDataFrame(data, ["product", "date", "price"])

# Define the window: Sort by Date (No partition needed if checking whole table)
timeWindow = Window.orderBy("date")

# Compare current 'price' to 'yesterday_price'
df_trends = df_prices.withColumn("yesterday_price", lag("price", 1).over(timeWindow)) \
              .withColumn("tomorrow_price", lead("price", 1).over(timeWindow))

display(df_trends)

In [0]:
from pyspark.sql.functions import rank, desc
from pyspark.sql.window import Window

# Create sample data with department and salary columns
data = [
    ("Aditya", "Engineering", 90000),
    ("Sita", "Engineering", 95000),
    ("Rahul", "Sales", 80000),
    ("Priya", "Sales", 85000)
]
df_employees = spark.createDataFrame(data, ["name", "department", "salary"])
display(df_employees)

# Define the window: Group by Dept, Sort by Salary
windowSpec = Window.partitionBy("department").orderBy(desc("salary"))

# Apply the rank
df_employees.withColumn("salary_rank", rank().over(windowSpec)).show()

In [0]:
from pyspark.sql.functions import posexplode

# Result: 
# Row 1: pos=0, project="Spark"
# Row 2: pos=1, project="Delta"
df.select("name", posexplode(col("projects")))
df.display()

In [0]:
from pyspark.sql.functions import create_map, lit, explode, col

# Creating a map of 'Location' -> 'Role'
df_map = df.withColumn("details", create_map(lit("Role"), lit("Data Engineer"), 
                                             lit("City"), lit("Bangalore")))

# First result: Show the map column
display(df_map.select("name", "details"))

# Second result: Explode the map into separate rows
df_map_flat = df_map.select("name", explode(col("details")).alias("attribute", "value"))

display(df_map_flat)


# First result: Show the map column
from pyspark.sql.functions import posexplode

# Result: 
# Row 1: pos=0, project="Spark"
# Row 2: pos=1, project="Delta"
df.select("name", posexplode(col("projects")))

In [0]:
data = [("Aditya", ["Spark", "Delta", "GenAI"])]
df = spark.createDataFrame(data, ["name", "projects"])

# Explode creates a row for every item in the 'projects' list
df_flat = df.select("name", explode(col("projects")).alias("project"))

display(df_flat)

In [0]:

%sql
-- Assuming a table named 'dev_table' exists with a 'skills' array column
SELECT 
  id,
  -- SQL syntax uses the arrow (->) operator
  TRANSFORM(clean_skills, s -> UPPER(s)) AS upper_skills,
  FILTER(clean_skills, s -> s LIKE 'S%') AS s_skills,
  EXISTS(clean_skills, s -> s = 'Python') AS knows_python
FROM main.default.csv_data1

In [0]:
df_csv_final.write.mode("append").saveAsTable("main.default.csv_data1")
 

 

# 1

In [0]:
# 1. Create a dummy CSV data (no file needed on serverless)
csv_str = "id,skills_str\n1,Spark;Python\n2,SQL;Tableau"

# 2. Read CSV directly from string using spark.read.csv with pandas StringIO workaround
import pandas as pd
from io import StringIO
df_pandas = pd.read_csv(StringIO(csv_str))
df_csv = spark.createDataFrame(df_pandas)

# 3. Split then Transform
from pyspark.sql.functions import split, col, transform, lower, filter, length

df_csv_final = df_csv.withColumn("skills_array", split(col("skills_str"), ";")) \
    .select(
        "id",
        # TRANSFORM: Convert array elements to lowercase
        transform("skills_array", lambda s: lower(s)).alias("clean_skills"),
        
        # FILTER: Only keep skills longer than 3 characters
        filter("skills_array", lambda s: length(s) > 3).alias("long_named_skills")
    )

display(df_csv_final)



In [0]:
"""Practice with JSON (The "Nested" Way)"""

from pyspark.sql.functions import transform, exists, concat, lit

# 1. Create DataFrame directly from JSON strings (no file needed)
json_str = """
{"id": 1, "info": {"tags": ["urgent", "tech", "billing"]}}
{"id": 2, "info": {"tags": ["low", "tech"]}}
"""

# 2. Read JSON directly from string using parallelize
from pyspark.sql import Row
import json
json_lines = [json.loads(line) for line in json_str.strip().split('\n')]
df_json = spark.createDataFrame(json_lines)

# Note: Access the nested array using dot notation inside the function
df_json_applied = df_json.select(
    "id",
    # TRANSFORM: Add a prefix to each tag
    transform("info.tags", lambda t: concat(lit("tag_"), t)).alias("prefixed_tags"),
    
    # EXISTS: Check if 'urgent' is in the nested tags
    exists("info.tags", lambda t: t == "urgent").alias("is_priority")
)

display(df_json_applied)

In [0]:
"""You said
TRANSFORM, FILTER, and EXISTS  Higher-Order Functions -- praticse code with handling csv , json and createdataframe"""

from pyspark.sql.functions import *
from pyspark.sql.types import *

# 1. Setup Data
data = [
    ("Aditya", ["Spark", "Python", "Java"]),
    ("Sita", ["SQL", "Excel"]),
    ("Rahul", ["spark", "pyspark", "MLflow"])
]
schema = "name STRING, skills ARRAY<STRING>"
df = spark.createDataFrame(data, schema)

# 2. Apply Higher-Order Functions
df_result = df.select(
    "name",
    # TRANSFORM: Convert all skills to uppercase
    transform("skills", lambda x: upper(x)).alias("upper_skills"),
    
    # FILTER: Keep only skills that start with 'S' (case-insensitive)
    filter("skills", lambda x: x.like("S%")).alias("s_skills"),
    
    # EXISTS: Boolean flag if they know 'Python'
    exists("skills", lambda x: x == "Python").alias("knows_python")
)

display(df_result)

In [0]:
# Path format: /Volumes/catalog/schema/volume/folder_name
output_volume_path = "/Volumes/main/default/raw_data_volume/exported_users_json"

# Write the data
""" df.write.format("json") \
  .mode("append") \
  .save(output_volume_path)"""
  

df.repartition(1).write.format("json") \
  .mode("overwrite") \
  .save("/Volumes/main/default/raw_data_volume/single_user_file")

print(f"Data successfully written to {output_volume_path}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.default

In [0]:
df.write.mode("overwrite").saveAsTable("database_dev.broze_taxi.user_records")

df.write.mode("overwrite").saveAsTable("main.default.json_data")

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, explode

# 1. Define the Schema (Best Practice)
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("info", StructType([
        StructField("role", StringType(), True),
        StructField("location", StringType(), True)
    ]), True),
    StructField("projects", ArrayType(StringType()), True),
    StructField("active", BooleanType(), True)
])

# 2. Create DataFrame directly from JSON strings (no file needed)
json_data = """[{"id": 1, "name": "Aditya", "info": {"role": "Data Engineer", "location": "Bangalore"}, "projects": ["ETL_Pipeline", "Delta_Migration"], "active": true},
{"id": 2, "name": "Sita", "info": {"role": "ML Engineer", "location": "Hyderabad"}, "projects": ["Model_Training"], "active": true},
{"id": 3, "name": "Rahul", "info": {"role": "Architect", "location": "Mumbai"}, "projects": ["Unity_Catalog", "GenAI_Bot", "Cloud_Security"], "active": false}]"""

# 3. Read JSON directly from string
import json
json_list = json.loads(json_data)
df = spark.createDataFrame(json_list, schema=schema)

display(df)

In [0]:
%sql
    
SHOW CATALOGS

show volumes LIKE 'database_dev.broze_taxi%

In [0]:
%sql
-- Usage
SELECT calculate_discount(100) AS discounted_price;

In [0]:
%sql
CREATE OR REPLACE FUNCTION database_dev.broze_taxi.calculate_discount(price DOUBLE)
RETURNS DOUBLE
RETURN price * 0.9;

In [0]:
{
  "user_id": 101,
  "profile": {
    "first_name": "Aditya",
    "skills": ["Spark", "Python", "SQL"]
  }
}

In [0]:
%sql
create schema if not exists database_dev.broze_taxi